# Simulation study I — LIT-MH-1 finding high-posterior models (paper §5.1)

Reproduces the LIT-MH-1 column of the paper's Table 1. The `DF_MCMC` defaults implement LIT-MH-1 exactly: add-weights clipped to $[p^{-1}, p]$, delete-weights to $[p^{-1}, 1]$, move probabilities $(h_a, h_d, h_s) = (0.4, 0.4, 0.2)$, 2,000 iterations.

**Data-generating model** (§5.1): rows of $X$ i.i.d. from $\mathrm{MN}(0, I_p)$ (independent design) or $\mathrm{MN}(0, \Sigma)$ with $\Sigma_{jk} = e^{-|j-k|}$ (correlated design); $y = X\beta^* + z$ with $z \sim \mathrm{MN}(0, I_n)$; the first 10 entries of $\beta^*$ are $\mathrm{SNR}\sqrt{\log p / n}\,(2, -3, 2, 2, -3, 3, -2, 3, -2, 3)$, the rest zero. Prior hyperparameters $\kappa_0 = 2$, $\kappa_1 = 3/2$. The chain starts from a random model of size 10.

**Metrics** (Table 1 style): $\hat\gamma_{\max}$ is the highest-posterior model sampled by the chain; $H_{\max}$ is the number of iterations needed to sample it (median, 95% quantile in brackets); $t_{\max}$ is the corresponding wall time, approximated by total time × first-visit iteration / total iterations. We also count how often $\hat\gamma_{\max} = \gamma_{\mathrm{true}}$.

For context, the paper's Table 1 shows RW-MH (100,000 iterations) needs $H_{\max}$ in the thousands to tens of thousands and fails on part of the correlated SNR = 2 runs, while LIT-MH-1 needs about 20 iterations — the theoretical floor, since $|\gamma^{(0)}| = |\gamma_{\mathrm{true}}| = 10$ and the two sets are disjoint with high probability. Increase `N_DATASETS` / edit `SETTINGS` for the full grid $(n, p) \in \{(500, 1000), (1000, 5000)\}$, SNR ∈ {1, 2, 3}.

In [ ]:
import time
import numpy as np
import pandas as pd

from BVS.DF_MCMC import DF_MCMC

In [ ]:
def simulate(n, p, design, SNR, seed):
    # paper sec. 5.1: independent design N(0, I_p) or correlated design
    # N(0, Sigma), Sigma_jk = exp(-|j-k|); y = X beta* + z, z ~ N(0, I_n)
    rng = np.random.default_rng(seed)
    if design == "independent":
        X = rng.standard_normal((n, p))
    elif design == "correlated":
        j = np.arange(p)
        Sigma = np.exp(-np.abs(j[:, None] - j[None, :]))
        X = rng.multivariate_normal(np.zeros(p), Sigma, size=n)
    else:
        raise ValueError(design)
    beta_star = np.zeros(p)
    beta_star[:10] = SNR * np.sqrt(np.log(p) / n) * np.array(
        [2, -3, 2, 2, -3, 3, -2, 3, -2, 3]
    )
    Y = X @ beta_star[:, None] + rng.standard_normal((n, 1))
    return X, Y, beta_star

In [ ]:
def log_post(keys, XTX, XTYr, YTY, g, kappa, N, P):
    # unnormalized log posterior (paper eq. (4)) of each model
    out = np.empty(len(keys))
    for i, key in enumerate(keys):
        pos = np.asarray(key)
        if pos.size == 0:
            fit = 0.0
        else:
            b = XTYr[pos]
            fit = float(b @ np.linalg.solve(XTX[np.ix_(pos, pos)], b))
        out[i] = (-kappa * pos.size * np.log(P)
                  - 0.5 * N * np.log((1 + 1 / g) * YTY - fit))
    return out


def unique_models(trace):
    # model (tuple of columns) -> first iteration it appears at
    seen = {}
    for t in range(trace.shape[1]):
        key = tuple(np.flatnonzero(trace[:, t]).tolist())
        if key not in seen:
            seen[key] = t
    return seen

In [ ]:
SETTINGS = [                          # paper grid also has (1000, 5000)
    (500, 1000, "independent", 3),    # and SNR in {1, 2, 3}
    (500, 1000, "correlated", 2),
]
N_DATASETS = 5                        # paper: 100
M_LIT = 2_000                         # paper Table 1 iteration count
KAPPA1, KAPPA2 = 1.5, 2.0             # paper: kappa_1 = 3/2, kappa_0 = 2

records = []
for (n, p, design, SNR) in SETTINGS:
    gmax_true, H, times, walls = 0, [], [], []
    for r in range(N_DATASETS):
        X, Y, beta_star = simulate(n, p, design, SNR, seed=1000 * r + 17)
        XTX = X.T @ X
        XTYr = (X.T @ Y).ravel()
        YTY = (Y.T @ Y).item()

        t0 = time.perf_counter()
        gamma_tr = DF_MCMC(Y, X, KAPPA1, KAPPA2, p, 10, M=M_LIT, burn_in=0,
                           seed=3000 * r + 11)
        wall = time.perf_counter() - t0
        lit_models = unique_models(gamma_tr)

        # gamma_max: highest-posterior model sampled by the chain
        keys = list(lit_models)
        lp = log_post(keys, XTX, XTYr, YTY, p ** (2 * KAPPA1) - 1,
                      KAPPA1 + KAPPA2, n, p)
        gmax = keys[int(np.argmax(lp))]
        gmax_true += int(gmax == tuple(range(10)))
        H.append(lit_models[gmax])
        times.append(wall * lit_models[gmax] / M_LIT)
        walls.append(wall)

    records.append(dict(
        setting=f"n={n}, p={p}, {design}, SNR={SNR}",
        iterations=M_LIT,
        time=float(np.mean(walls)),
        gmax_is_gtrue=f"{gmax_true}/{N_DATASETS}",
        H_max=f"{np.median(H):.0f} ({np.quantile(H, 0.95):.0f})",
        t_max=float(np.median(times)),
    ))
    print(f"n={n}, p={p}, {design}, SNR={SNR}: gmax = gamma_true in "
          f"{gmax_true}/{N_DATASETS} datasets, H_max median {np.median(H):.0f}")

In [ ]:
df = pd.DataFrame(records)
df

## Notes

- $H_{\max}$ for LIT-MH-1 should be close to **20**, the theoretical floor: $|\gamma^{(0)}| = |\gamma_{\mathrm{true}}| = 10$ and the two sets are disjoint with high probability, so at least 10 deletions plus 10 additions are needed (the paper §5.1 reports medians of about 20 and calls this near-optimal).
- At SNR = 3, $\hat\gamma_{\max}$ usually equals $\gamma_{\mathrm{true}}$; at SNR = 1 the best model is often the null model, which the chain finds trivially (not included in the default `SETTINGS`).
- For the correlated design at SNR = 2 the posterior is multi-modal; the chain may need a few hundred iterations and occasionally does not find the best model within 2,000 iterations (the paper reports 79–89 successes out of 100 for the informed samplers in this setting).